In [1]:
import duckdb
import time
from pathlib import Path

DB_PATH = "./mimiciv.duckdb"
MIMIC_PATH = Path(
    "physionet.org/files/mimiciv/3.1"
)  # adjust if your raw files live elsewhere

HOSP = MIMIC_PATH / "hosp"
ICU = MIMIC_PATH / "icu"

TABLES = {
    # Hosp module
    "hosp.patients": HOSP / "patients.csv.gz",
    "hosp.admissions": HOSP / "admissions.csv.gz",
    "hosp.transfers": HOSP / "transfers.csv.gz",
    "hosp.diagnoses_icd": HOSP / "diagnoses_icd.csv.gz",
    "hosp.d_icd_diagnoses": HOSP / "d_icd_diagnoses.csv.gz",
    "hosp.procedures_icd": HOSP / "procedures_icd.csv.gz",
    "hosp.d_icd_procedures": HOSP / "d_icd_procedures.csv.gz",
    "hosp.labevents": HOSP / "labevents.csv.gz",
    "hosp.d_labitems": HOSP / "d_labitems.csv.gz",
    "hosp.prescriptions": HOSP / "prescriptions.csv.gz",
    "hosp.microbiologyevents": HOSP / "microbiologyevents.csv.gz",
    "hosp.services": HOSP / "services.csv.gz",
    "hosp.drgcodes": HOSP / "drgcodes.csv.gz",
    "hosp.omr": HOSP / "omr.csv.gz",
    # ICU module
    "icu.icustays": ICU / "icustays.csv.gz",
    "icu.chartevents": ICU / "chartevents.csv.gz",
    "icu.d_items": ICU / "d_items.csv.gz",
    "icu.inputevents": ICU / "inputevents.csv.gz",
    "icu.outputevents": ICU / "outputevents.csv.gz",
    "icu.procedureevents": ICU / "procedureevents.csv.gz",
    "icu.datetimeevents": ICU / "datetimeevents.csv.gz",
}


def load_tables(con):
    con.execute("CREATE SCHEMA IF NOT EXISTS hosp")
    con.execute("CREATE SCHEMA IF NOT EXISTS icu")
    for table_name, file_path in TABLES.items():
        if not file_path.exists():
            print(f"  SKIP {table_name}: file not found at {file_path}")
            continue
        print(f"  Loading {table_name}...", end=" ", flush=True)
        start = time.time()
        con.execute(f"DROP TABLE IF EXISTS {table_name}")
        con.execute(f"""
            CREATE TABLE {table_name} AS
            SELECT * FROM read_csv_auto('{file_path}', header=True)
        """)
        n_rows = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        elapsed = time.time() - start
        print(f"{n_rows:,} rows in {elapsed:.1f}s")


def create_views(con):
    con.execute("""
        CREATE OR REPLACE VIEW patient_admissions AS
        SELECT
            p.subject_id, p.gender, p.anchor_age, p.anchor_year, p.dod,
            a.hadm_id, a.admittime, a.dischtime, a.deathtime,
            a.admission_type, a.admission_location, a.discharge_location,
            a.insurance, a.language, a.marital_status, a.race,
            a.hospital_expire_flag,
            EXTRACT(EPOCH FROM (a.dischtime - a.admittime)) / 3600 AS hospital_los_hours
        FROM hosp.patients p
        JOIN hosp.admissions a ON p.subject_id = a.subject_id
    """)
    con.execute("""
        CREATE OR REPLACE VIEW icu_stays_full AS
        SELECT
            i.stay_id, i.subject_id, i.hadm_id,
            i.first_careunit, i.last_careunit, i.intime, i.outtime,
            EXTRACT(EPOCH FROM (i.outtime - i.intime)) / 3600 AS icu_los_hours,
            pa.gender, pa.anchor_age, pa.race, pa.insurance,
            pa.admission_type, pa.hospital_expire_flag, pa.dod
        FROM icu.icustays i
        JOIN patient_admissions pa ON i.hadm_id = pa.hadm_id
    """)
    con.execute("""
        CREATE OR REPLACE VIEW cam_icu_assessments AS
        SELECT
            ce.subject_id, ce.hadm_id, ce.stay_id, ce.charttime, ce.itemid,
            di.label AS item_label, ce.value, ce.valuenum,
            CASE 
                WHEN ce.value IN ('Positive', 'Yes', 'positive', 'yes') THEN 1
                WHEN ce.value IN ('Negative', 'No', 'negative', 'no') THEN 0
                ELSE NULL
            END AS cam_positive
        FROM icu.chartevents ce
        JOIN icu.d_items di ON ce.itemid = di.itemid
        WHERE LOWER(di.label) LIKE '%cam%'
           OR LOWER(di.label) LIKE '%delirium%'
           OR LOWER(di.label) LIKE '%confusion assessment%'
    """)
    con.execute("""
        CREATE OR REPLACE VIEW delirium_summary AS
        SELECT
            stay_id,
            COUNT(*) AS total_assessments,
            SUM(cam_positive) AS positive_assessments,
            MAX(cam_positive) AS ever_delirious,
            MIN(CASE WHEN cam_positive = 1 THEN charttime END) AS first_positive_time,
            MIN(charttime) AS first_assessment_time,
            MAX(charttime) AS last_assessment_time
        FROM cam_icu_assessments
        WHERE cam_positive IS NOT NULL
        GROUP BY stay_id
    """)
    con.execute("""
        CREATE OR REPLACE VIEW diagnoses_labeled AS
        SELECT
            d.subject_id, d.hadm_id, d.seq_num, d.icd_code, d.icd_version,
            dd.long_title AS diagnosis
        FROM hosp.diagnoses_icd d
        LEFT JOIN hosp.d_icd_diagnoses dd 
            ON d.icd_code = dd.icd_code AND d.icd_version = dd.icd_version
    """)
    con.execute("""
        CREATE OR REPLACE VIEW labs_labeled AS
        SELECT
            le.subject_id, le.hadm_id, le.charttime, le.itemid,
            dl.label AS lab_name, dl.category AS lab_category,
            le.value, le.valuenum, le.valueuom AS unit, le.flag
        FROM hosp.labevents le
        JOIN hosp.d_labitems dl ON le.itemid = dl.itemid
    """)


# Build only if the database doesn't exist; otherwise just connect
if Path(DB_PATH).exists():
    print(f"Database already exists at {DB_PATH}, connecting...")
    con = duckdb.connect(database=DB_PATH)
else:
    print(f"No database found at {DB_PATH}, building from raw files...")
    if not MIMIC_PATH.exists():
        raise FileNotFoundError(
            f"Raw MIMIC-IV files not found at {MIMIC_PATH}. "
            f"Update MIMIC_PATH at the top of this cell to point to your data."
        )
    con = duckdb.connect(database=DB_PATH)
    print("\n>>> Loading tables...")
    load_tables(con)
    print("\n>>> Creating views...")
    create_views(con)
    print(f"\nDatabase built at {DB_PATH}")

print("Connection ready.")

Database already exists at ./mimiciv.duckdb, connecting...
Connection ready.


In [30]:
con.execute("""
    SELECT COUNT(*) AS icu_stays_under_18
    FROM icu_stays_full
    WHERE anchor_age < 18
""").fetchdf()

,icu_stays_under_18
0,0


In [31]:
con.execute("""
    SELECT
        stay_id,
        subject_id,
        hadm_id,
        intime,
        outtime,
        CASE
            WHEN intime IS NULL AND outtime IS NULL THEN 'NULL intime + NULL outtime'
            WHEN intime IS NULL                     THEN 'NULL intime'
            WHEN outtime IS NULL                    THEN 'NULL outtime'
            WHEN outtime <= intime                  THEN 'outtime <= intime'
        END AS issue
    FROM icu.icustays
    WHERE intime IS NULL
       OR outtime IS NULL
       OR outtime <= intime
    ORDER BY stay_id
""").fetchdf()

,stay_id,subject_id,hadm_id,intime,outtime,issue
0,30924165,10882284,28910170,2125-04-23 03:13:54,NaT,NULL outtime
1,31619283,14330929,27393728,2188-06-12 16:22:17,NaT,NULL outtime
2,33009457,16117624,26745154,2180-08-17 17:29:29,NaT,NULL outtime
3,33734563,16799689,26464023,2166-04-08 13:46:41,NaT,NULL outtime
4,34314756,18717462,26658752,2189-03-05 16:50:37,NaT,NULL outtime
5,34971926,19526758,27199762,2153-07-02 08:35:23,NaT,NULL outtime
6,34986686,16348177,22166548,2151-10-05 18:47:26,NaT,NULL outtime
7,36353950,15882332,24693580,2178-03-27 19:18:14,NaT,NULL outtime
8,37339160,11661851,24632782,2173-01-17 18:24:31,NaT,NULL outtime
9,37403535,15777534,23340499,2126-03-09 22:46:50,NaT,NULL outtime


In [2]:
con.execute("""
    SELECT
        ever_delirious,
        COUNT(*) AS n_stays
    FROM delirium_summary
    GROUP BY ever_delirious
    ORDER BY ever_delirious
""").df()

,ever_delirious,n_stays
0,0,39046
1,1,33403
